# 06 · Quantization: int8, int4 groups, NF4, FP8, FP4 (MXFP4 / NVFP4)

**Needed for:** quantization papers (LLM track, step 4), FP8 training in DeepSeek-V3 (step 7), and the FP4 KV
cache in DeepSeek-V4.1-Flash (step 10).

Every scheme here is the same recipe with different choices:

1. Pick a **grid** of representable values (evenly spaced integers, normal-distribution quantiles, or floats).
2. Pick a **scale granularity**: one scale per tensor, per row/channel, per group of 32–128 values, or per 2-D tile.
3. Quantize: `q = round_to_grid(x / scale)`. Dequantize: `x̂ = q * scale`.

Smaller groups track local magnitudes better (less error) but store more scales.
All code below *simulates* quantization in fp32 so you can measure error on any hardware.

In [ ]:
import torch

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")


def rel_err(x, x_hat):
    return ((x - x_hat).norm() / x.norm()).item()

## 1. Symmetric int8: per-tensor vs per-channel

`scale = absmax / 127`. One outlier makes the per-tensor scale large, so every other value is rounded coarsely.
Real LLM activations have a few outlier channels with values 20–100× larger than the rest (LLM.int8 paper),
which is exactly this failure.

In [ ]:
def quant_int8(x, dim=None):
    """Symmetric absmax int8. dim=None: one scale for the tensor. dim=-1: one scale per row."""
    amax = x.abs().amax() if dim is None else x.abs().amax(dim=dim, keepdim=True)
    scale = amax.clamp(min=1e-12) / 127
    q = (x / scale).round().clamp(-127, 127).to(torch.int8)
    return q, scale


def dequant(q, scale):
    return q.float() * scale


W = torch.randn(256, 256)
W_out = W.clone()
W_out[3, 7] = 60.0   # a single outlier

for name, w in (("no outlier", W), ("one outlier", W_out)):
    e_tensor = rel_err(w, dequant(*quant_int8(w)))
    e_row = rel_err(w, dequant(*quant_int8(w, dim=-1)))
    print(f"{name:12s} per-tensor err {e_tensor:.4f}   per-row err {e_row:.4f}")

## 2. int4 with groups, symmetric and asymmetric

int4 has only 16 levels, so granularity matters much more. Weight-only methods (GPTQ, AWQ) use a scale per
group of 64–128 input channels. **Asymmetric** quantization adds a zero point, which helps when values are not
centred on zero (after a ReLU, for example).

In [ ]:
def quant_int4_group(w, group=128, symmetric=True):
    """w: (out, in) with in divisible by group. Returns the dequantized tensor (simulation)."""
    out_f, in_f = w.shape
    g = w.reshape(out_f, in_f // group, group)
    if symmetric:
        scale = g.abs().amax(-1, keepdim=True).clamp(min=1e-12) / 7
        q = (g / scale).round().clamp(-8, 7)
        return (q * scale).reshape(out_f, in_f)
    lo, hi = g.amin(-1, keepdim=True), g.amax(-1, keepdim=True)
    scale = (hi - lo).clamp(min=1e-12) / 15
    zero = (-lo / scale).round()
    q = (g / scale + zero).round().clamp(0, 15)
    return ((q - zero) * scale).reshape(out_f, in_f)


def test_int4(fn):
    w = torch.randn(64, 256)
    for group in (32, 128):
        w_hat = fn(w, group=group)
        levels = [w_hat[0, i:i + group].unique().numel() for i in range(0, 256, group)]
        assert max(levels) <= 16, "at most 16 distinct values per group"
        assert rel_err(w, w_hat) < 0.2
    assert rel_err(w, fn(w, group=32)) < rel_err(w, fn(w, group=256))  # smaller groups, lower error


test_int4(quant_int4_group)
for group in (32, 128, 256):
    print(f"group {group:3d}: symmetric err {rel_err(W, quant_int4_group(W, group)):.4f}"
          f"   asymmetric err {rel_err(W, quant_int4_group(W, group, symmetric=False)):.4f}")
relu_w = W.clamp(min=0)
print(f"after ReLU (all >= 0): symmetric {rel_err(relu_w, quant_int4_group(relu_w)):.4f}"
      f"   asymmetric {rel_err(relu_w, quant_int4_group(relu_w, symmetric=False)):.4f}")

## 3. NF4 (QLoRA): a grid shaped like the weights

Trained weights are roughly normally distributed, so evenly spaced levels waste resolution in the tails.
NF4's 16 levels are quantiles of a normal distribution, scaled to [-1, 1], with an exact zero.
Each block of 64 weights stores one absmax scale. (QLoRA then quantizes those scales too: "double quantization".)

In [ ]:
NF4 = torch.tensor([
    -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
    -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
    0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
    0.7229568362236023, 1.0,
])
INT4_SYM = torch.linspace(-1, 1, 16)  # evenly spaced 16 levels in [-1, 1], for a fair comparison


def quant_to_grid(x, grid, block=64):
    """Absmax-scale each block to [-1, 1], snap to the nearest grid value, scale back."""
    flat = x.reshape(-1, block)
    scale = flat.abs().amax(-1, keepdim=True).clamp(min=1e-12)
    idx = ((flat / scale)[..., None] - grid).abs().argmin(-1)   # nearest level; idx fits in 4 bits
    return (grid[idx] * scale).reshape(x.shape)


w = torch.randn(1024, 1024)
print(f"normal weights, block 64: uniform 4-bit err {rel_err(w, quant_to_grid(w, INT4_SYM)):.4f}"
      f"   NF4 err {rel_err(w, quant_to_grid(w, NF4)):.4f}")

## 4. FP8: E4M3 and E5M2

| Format | Max value | Precision | Typical use |
|---|---|---|---|
| E4M3 | 448 | 3 mantissa bits | weights and activations (forward) |
| E5M2 | 57344 | 2 mantissa bits | gradients (need range more than precision) |

Being floating point, FP8 has *relative* precision: small and large values get similar relative error, unlike
int8. But the range is tiny compared to bf16, so you still need a scale.

**DeepSeek-V3's recipe:** activations get one scale per 1×128 tile (per token, per 128 channels) and weights one
scale per 128×128 block. Fine-grained scales absorb outliers without per-tensor range problems. PyTorch has
`torch.float8_e4m3fn` / `torch.float8_e5m2` dtypes, so casting simulates real rounding.

In [ ]:
F8 = torch.float8_e4m3fn
F8_MAX = torch.finfo(F8).max
print("E4M3 max:", F8_MAX, "| E5M2 max:", torch.finfo(torch.float8_e5m2).max)


def fp8_per_tensor(x):
    scale = x.abs().amax() / F8_MAX
    return (x / scale).to(F8).float() * scale


def fp8_blockwise(x, tile=128):
    """One scale per (1 x tile) slice along the last dim, as DeepSeek-V3 does for activations."""
    rows, cols = x.shape
    g = x.reshape(rows, cols // tile, tile)
    scale = g.abs().amax(-1, keepdim=True).clamp(min=1e-12) / F8_MAX
    return ((g / scale).to(F8).float() * scale).reshape(rows, cols)


act = torch.randn(512, 1024)
act[:, 17] *= 200     # an outlier channel, like real LLM activations
act[5] *= 0.001       # and a token with tiny activations
print(f"activations with outliers: per-tensor FP8 err {rel_err(act, fp8_per_tensor(act)):.4f}   "
      f"1x128 blockwise err {rel_err(act, fp8_blockwise(act)):.4f}")
print(f"the tiny token alone:      per-tensor {rel_err(act[5], fp8_per_tensor(act)[5]):.4f}   "
      f"blockwise {rel_err(act[5], fp8_blockwise(act)[5]):.4f}")

## 5. FP4 (E2M1) with block scales: MXFP4 and NVFP4

E2M1 can only represent ±{0, 0.5, 1, 1.5, 2, 3, 4, 6}. Everything depends on the block scale:

| Format | Block size | Scale type |
|---|---|---|
| MXFP4 (OCP standard) | 32 | power of two (E8M0: exponent only) |
| NVFP4 (NVIDIA Blackwell) | 16 | FP8 E4M3, plus one fp32 scale per tensor |

A power-of-two scale is cheap in hardware but can waste up to half the range. NVFP4's finer, non-power-of-two
scale fits the block's absmax to 6 exactly. This is the kind of format behind "FP4 KV cache" in V4.1-Flash.

In [ ]:
E2M1 = torch.tensor([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])
E2M1 = torch.cat([-E2M1.flip(0)[:-1], E2M1])   # 15 values: -6 ... 6


def round_to_e2m1(x):
    return E2M1[(x[..., None] - E2M1).abs().argmin(-1)]


def mxfp4(x, block=32):
    g = x.reshape(-1, block)
    amax = g.abs().amax(-1, keepdim=True).clamp(min=1e-30)
    scale = 2.0 ** (torch.floor(torch.log2(amax)) - 2)   # E2M1's largest exponent is 2 (6 = 1.5 * 2^2)
    return (round_to_e2m1((g / scale).clamp(-6, 6)) * scale).reshape(x.shape)


def nvfp4(x, block=16):
    g = x.reshape(-1, block)
    scale = g.abs().amax(-1, keepdim=True).clamp(min=1e-30) / 6
    scale = scale.to(F8).float()       # the block scale itself is stored in FP8 E4M3
    return (round_to_e2m1((g / scale).clamp(-6, 6)) * scale).reshape(x.shape)


def test_e2m1(fn):
    x = torch.tensor([0.2, 0.3, 0.74, 0.76, 2.4, 2.6, 5.1, 5.9, -1.3, 100.0])
    torch.testing.assert_close(fn(x.clamp(-6, 6)), torch.tensor([0.0, 0.5, 0.5, 1.0, 2.0, 3.0, 6.0, 6.0, -1.5, 6.0]))


test_e2m1(round_to_e2m1)
kv = torch.randn(4096, 128)
print(f"KV-like tensor: MXFP4 err {rel_err(kv, mxfp4(kv)):.4f}   NVFP4 err {rel_err(kv, nvfp4(kv)):.4f}   "
      f"FP8 blockwise err {rel_err(kv, fp8_blockwise(kv)):.4f}")

Ties like 0.25 (exactly between 0 and 0.5) are a detail: hardware uses round-to-nearest-even, `argmin` picks
the first. The test avoids exact ties.

## 6. How quantized weights are actually used

- **Weight-only (W4A16):** store int4 weights, dequantize to bf16 inside the matmul kernel. Helps decoding, which
  is memory-bound: you read 4× fewer weight bytes. Doesn't speed up compute.
- **Weight + activation (W8A8, FP8, FP4):** the matmul itself runs in low precision on tensor cores. Helps
  compute-bound prefill and training.
- **GPTQ** chooses the rounding of each weight to compensate for the error of weights already rounded (using
  second-order information from calibration data). **AWQ** scales up the few input channels with large
  activations before quantizing so their weights keep precision.

## Drill

In [ ]:
def my_quant_int4_group(w, group=128, symmetric=True):
    """Return the dequantized tensor. Symmetric path is enough to pass."""
    raise NotImplementedError


def my_round_to_e2m1(x):
    raise NotImplementedError


drill(test_int4, my_quant_int4_group)
drill(test_e2m1, my_round_to_e2m1)